In [1]:
"""
FILE 1: feature_Engineering.ipynb
==================================
Reads raw CSVs → applies full feature engineering → saves output files.
Run this FIRST, then run model_training.ipynb.

Inputs  (in ../needed/):
  insurance_dataset.csv   — 510 customers
  insurance_claims.csv    — 1000 claim records

Outputs (in visualization/ folder):
  customer_engineered.csv
  claims_engineered.csv
  claims_aggregated.csv
  engineering_metadata.pkl
"""

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("FEATURE ENGINEERING PIPELINE")
print("=" * 60)

FEATURE ENGINEERING PIPELINE


In [2]:
# ════════════════════════════════════════════════════════════════
# STEP 1: LOAD RAW DATA
# ════════════════════════════════════════════════════════════════
customer_df = pd.read_csv("../needed/insurance_dataset.csv")
claims_df   = pd.read_csv("../needed/insurance_claims.csv")

print(f"Loaded: customers {customer_df.shape}  |  claims {claims_df.shape}")
print()
print("Customer columns:", customer_df.columns.tolist())
print()
print("Customer sample:")
print(customer_df.head(3).to_string())

Loaded: customers (510, 9)  |  claims (1000, 40)

Customer columns: ['Customer_ID', 'Age', 'Gender', 'Policy_Type', 'Premium', 'Claim_Count', 'Region', 'Date_Joined', 'Customer_Satisfaction']

Customer sample:
  Customer_ID  Age  Gender Policy_Type  Premium  Claim_Count Region Date_Joined  Customer_Satisfaction
0    CUST0001   26  Female        Life    57.78            3  North  2019-09-20                    1.0
1    CUST0002   22       M        Life   243.23            3  South  2021-11-21                   10.0
2    CUST0003   28     NaN        Auto   341.58            2  North  2020-09-10                    3.0


In [3]:
# ════════════════════════════════════════════════════════════════
# STEP 2: CLEAN CUSTOMER DATA
# ════════════════════════════════════════════════════════════════
df = customer_df.copy()

# Normalise gender values  (M→Male, F→Female, blanks→Unknown)
gender_map = {'M': 'Male', 'F': 'Female', 'Male': 'Male',
              'Female': 'Female', 'Other': 'Other'}
df['Gender'] = df['Gender'].map(gender_map).fillna('Unknown')

# Impute Premium by Policy_Type median, Satisfaction by global median
df['Premium'] = df.groupby('Policy_Type')['Premium'].transform(
    lambda x: x.fillna(x.median()))
df['Customer_Satisfaction'] = df['Customer_Satisfaction'].fillna(
    df['Customer_Satisfaction'].median())

print("Missing after imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print("Gender distribution:", df['Gender'].value_counts().to_dict())
print("Policy_Type distribution:", df['Policy_Type'].value_counts().to_dict())
print("Region distribution:", df['Region'].value_counts().to_dict())

Missing after imputation:
Series([], dtype: int64)

Gender distribution: {'Female': 202, 'Male': 190, 'Other': 93, 'Unknown': 25}
Policy_Type distribution: {'Life': 174, 'Auto': 168, 'Home': 168}
Region distribution: {'South': 140, 'North': 134, 'West': 129, 'East': 107}


In [4]:
# ════════════════════════════════════════════════════════════════
# STEP 3: ENGINEER CUSTOMER FEATURES
# ════════════════════════════════════════════════════════════════

# Tenure
df['Date_Joined']  = pd.to_datetime(df['Date_Joined'])
reference_date     = pd.Timestamp('2025-01-01')
df['tenure_years'] = (reference_date - df['Date_Joined']).dt.days / 365.25

# Age group  (matching actual data range 18–79)
df['age_group'] = pd.cut(df['Age'], bins=[0, 35, 55, 100],
                          labels=['Young', 'Mid', 'Senior'])

# Premium tier (quantile-based so each bin has equal customers)
df['premium_tier'] = pd.qcut(df['Premium'], q=3,
                               labels=['Low', 'Mid', 'High'],
                               duplicates='drop')

# Interaction / ratio features
df['claims_per_year']         = df['Claim_Count'] / df['tenure_years'].replace(0, 0.1)
df['age_x_claims']            = df['Age'] * df['Claim_Count']
df['premium_x_satisfaction']  = df['Premium'] * df['Customer_Satisfaction']
df['satisfaction_per_tenure'] = df['Customer_Satisfaction'] / df['tenure_years'].replace(0, 0.1)

# Satisfaction group  (scale 1–10: Low ≤3, Mid 4–7, High >7)
df['satisfaction_group'] = pd.cut(df['Customer_Satisfaction'],
                                    bins=[0, 3, 7, 10],
                                    labels=['Low', 'Mid', 'High'])

# High-value customer flag (top 25% premium)
premium_75th        = df['Premium'].quantile(0.75)
df['is_high_value'] = (df['Premium'] > premium_75th).astype(int)

# ── Churn Risk label ─────────────────────────────────────────────
# A customer is at churn risk if:
#   • satisfaction ≤ 5  (below median on 1–10 scale), OR
#   • claims_per_year ≥ 2 (filing many claims), OR
#   • tenure < 1.5 years  (very new, not yet retained)
churn_mask = (
    (df['Customer_Satisfaction'] <= 5) |
    (df['claims_per_year'] >= 2) |
    (df['tenure_years'] < 1.5)
)
df['churn_risk'] = churn_mask.astype(int)

# ── Loyalty label ────────────────────────────────────────────────
# Loyal customer: long tenure AND high satisfaction AND low claim rate
loyalty_mask = (
    (df['tenure_years'] >= 3) &
    (df['Customer_Satisfaction'] >= 7) &
    (df['claims_per_year'] <= 1)
)
df['is_loyal'] = loyalty_mask.astype(int)

print(f"premium_75th threshold : {premium_75th:.2f}")
print(f"High-value customers   : {df['is_high_value'].sum()} / {len(df)}")
print(f"Churn-risk customers   : {df['churn_risk'].sum()} ({df['churn_risk'].mean()*100:.1f}%)")
print(f"Loyal customers        : {df['is_loyal'].sum()} ({df['is_loyal'].mean()*100:.1f}%)")

premium_75th threshold : 378.68
High-value customers   : 128 / 510
Churn-risk customers   : 307 (60.2%)
Loyal customers        : 120 (23.5%)


In [5]:
# ════════════════════════════════════════════════════════════════
# STEP 4: ONE-HOT ENCODE CUSTOMER CATEGORICALS
# ════════════════════════════════════════════════════════════════
ohe_cols   = ['Gender', 'Region', 'age_group', 'premium_tier', 'satisfaction_group']
df_encoded = pd.get_dummies(df, columns=ohe_cols, drop_first=False)
df_encoded.drop(columns=['Date_Joined'], inplace=True)

print(f"Customer engineered shape: {df_encoded.shape}")
print(f"Columns: {df_encoded.columns.tolist()}")

Customer engineered shape: (510, 31)
Columns: ['Customer_ID', 'Age', 'Policy_Type', 'Premium', 'Claim_Count', 'Customer_Satisfaction', 'tenure_years', 'claims_per_year', 'age_x_claims', 'premium_x_satisfaction', 'satisfaction_per_tenure', 'is_high_value', 'churn_risk', 'is_loyal', 'Gender_Female', 'Gender_Male', 'Gender_Other', 'Gender_Unknown', 'Region_East', 'Region_North', 'Region_South', 'Region_West', 'age_group_Young', 'age_group_Mid', 'age_group_Senior', 'premium_tier_Low', 'premium_tier_Mid', 'premium_tier_High', 'satisfaction_group_Low', 'satisfaction_group_Mid', 'satisfaction_group_High']


In [6]:
# ════════════════════════════════════════════════════════════════
# STEP 5: ENGINEER CLAIMS FEATURES
# ════════════════════════════════════════════════════════════════
cl = claims_df.copy()

# Drop junk column and replace '?' with NaN
cl.drop(columns=['_c39'], errors='ignore', inplace=True)
cl.replace('?', np.nan, inplace=True)

# Derived numeric features
severity_map = {'Trivial Damage': 1, 'Minor Damage': 2,
                'Major Damage':   3, 'Total Loss':    4}
cl['severity_score']    = cl['incident_severity'].map(severity_map)
cl['is_fraud']          = (cl['fraud_reported'] == 'Y').astype(int)
cl['is_night_incident'] = cl['incident_hour_of_the_day'].apply(
    lambda h: 1 if pd.notnull(h) and (int(h) >= 22 or int(h) <= 6) else 0)
cl['is_collision']      = cl['incident_type'].apply(
    lambda x: 1 if pd.notnull(x) and 'Collision' in str(x) else 0)
cl['vehicle_age']       = 2015 - cl['auto_year'].fillna(2005).astype(float)
cl['claim_per_premium'] = (
    cl['total_claim_amount'] / cl['policy_annual_premium'].replace(0, np.nan))
cl['injury_ratio']      = cl['injury_claim']   / cl['total_claim_amount'].replace(0, np.nan)
cl['vehicle_ratio']     = cl['vehicle_claim']  / cl['total_claim_amount'].replace(0, np.nan)
cl['property_ratio']    = cl['property_claim'] / cl['total_claim_amount'].replace(0, np.nan)

# Aggregate claims per policy (useful for joining with customer data if needed)
claims_agg = cl.groupby('policy_number').agg(
    total_claim_amount    = ('total_claim_amount',          'sum'),
    avg_injury_claim      = ('injury_claim',                'mean'),
    avg_property_claim    = ('property_claim',              'mean'),
    avg_vehicle_claim     = ('vehicle_claim',               'mean'),
    fraud_rate            = ('is_fraud',                    'mean'),
    has_fraud_history     = ('is_fraud',                    'max'),
    avg_severity          = ('severity_score',              'mean'),
    max_severity          = ('severity_score',              'max'),
    total_incidents       = ('incident_type',               'count'),
    night_incident_rate   = ('is_night_incident',           'mean'),
    collision_rate        = ('is_collision',                'mean'),
    avg_vehicles_involved = ('number_of_vehicles_involved', 'mean'),
    avg_bodily_injuries   = ('bodily_injuries',             'mean'),
    avg_witnesses         = ('witnesses',                   'mean'),
    annual_premium        = ('policy_annual_premium',       'mean'),
).reset_index()
claims_agg['claim_to_premium_ratio'] = (
    claims_agg['total_claim_amount'] / claims_agg['annual_premium'].replace(0, np.nan))
claims_agg['risk_score'] = (
    claims_agg['fraud_rate'] * 2 +
    claims_agg['avg_severity'] +
    claims_agg['night_incident_rate'])

# One-hot encode categorical claim columns
cat_cols = ['insured_sex', 'insured_education_level', 'insured_relationship',
            'incident_type', 'collision_type', 'incident_severity',
            'authorities_contacted', 'policy_state', 'policy_csl',
            'insured_occupation', 'property_damage', 'police_report_available']
cl_encoded = pd.get_dummies(cl, columns=[c for c in cat_cols if c in cl.columns],
                             drop_first=True)

print(f"Claims aggregated shape  : {claims_agg.shape}")
print(f"Claims engineered shape  : {cl_encoded.shape}")
print(f"Fraud rate in dataset    : {cl['is_fraud'].mean()*100:.1f}%")

Claims aggregated shape  : (1000, 18)
Claims engineered shape  : (1000, 78)
Fraud rate in dataset    : 24.7%


In [7]:
# ════════════════════════════════════════════════════════════════
# STEP 6: SAVE ALL OUTPUTS
# ════════════════════════════════════════════════════════════════

df_encoded.to_csv("customer_engineered.csv", index=False)
print("Saved: customer_engineered.csv", df_encoded.shape)

claims_agg.to_csv("claims_aggregated.csv", index=False)
print("Saved: claims_aggregated.csv", claims_agg.shape)

cl_encoded.to_csv("claims_engineered.csv", index=False)
print("Saved: claims_engineered.csv", cl_encoded.shape)

# Metadata — everything model_training.ipynb needs to stay consistent
metadata = {
    'premium_75th'     : float(premium_75th),
    'reference_date'   : str(reference_date),
    'ohe_cols'         : ohe_cols,
    'severity_map'     : severity_map,
    'customer_shape'   : df_encoded.shape,
    'claims_shape'     : cl_encoded.shape,
    'customer_columns' : df_encoded.columns.tolist(),
    'claims_columns'   : cl_encoded.columns.tolist(),
}
with open("engineering_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Saved: engineering_metadata.pkl")

print()
print("=" * 60)
print("Feature engineering complete. Run model_training.ipynb next.")
print("=" * 60)

Saved: customer_engineered.csv (510, 31)
Saved: claims_aggregated.csv (1000, 18)
Saved: claims_engineered.csv (1000, 78)
Saved: engineering_metadata.pkl

Feature engineering complete. Run model_training.ipynb next.
